# film revenue prediction - project notebook

## sections
1. project overview
2. data acquisition
3. data merging
4. data cleaning
5. feature engineering
6. data leakage analysis
7. ablation study
8. fine-tuning experiment
9. interpretability with shap
10. evaluation metrics
11. recommended models + tuning
12. pipeline summary and timeline
13. report-ready outputs
14. pitfalls checklist

In [ ]:
# setup
import os
import json
import numpy as np
import pandas as pd

from pathlib import Path

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
np.random.seed(SEED)

## 1) project overview

### 1.1 central research question
- what matters more for pre-release prediction: feature representation or model choice?

### 1.2 prediction target
- primary target: `y = log1p(revenue)`
- derived label: `profitable = 1[revenue > 1.5 * budget]`

### 1.3 core thesis
- combining structured metadata + synopsis embeddings + poster embeddings should outperform any single modality.

## 2) data acquisition

### 2.1 tmdb (kaggle)
- `TMDB_movie_dataset_v11.csv`

### 2.2 imdb non-commercial files
- `title.crew.tsv.csv` (gzipped)
- `title.principals.tsv.csv` (gzipped)
- `name.basics.tsv.csv` (gzipped)

### 2.3 poster images
- build url: `https://image.tmdb.org/t/p/w342/{poster_path}`
- cache locally before embedding extraction

In [ ]:
# load raw datasets
TMDB_PATH       = ROOT / "TMDB_movie_dataset_v11.csv"
CREW_PATH       = ROOT / "title.crew.tsv.csv"
PRINCIPALS_PATH = ROOT / "title.principals.tsv.csv"
NAMES_PATH      = ROOT / "name.basics.tsv.csv"

tmdb       = pd.read_csv(TMDB_PATH)
crew       = pd.read_csv(CREW_PATH,       sep='\t', compression='gzip', low_memory=False)
principals = pd.read_csv(PRINCIPALS_PATH, sep='\t', compression='gzip', low_memory=False)
names      = pd.read_csv(NAMES_PATH,      sep='\t', compression='gzip', low_memory=False)

print(f"tmdb:       {tmdb.shape}")
print(f"crew:       {crew.shape}")
print(f"principals: {principals.shape}")
print(f"names:      {names.shape}")
tmdb.head()

## 3) data merging

- use tmdb as base table
- join tmdb `imdb_id` to imdb `tconst`
- keep join diagnostics: row counts and match rate

> note: TMDB v11 has ~1.4M rows including TV/shorts — low overall match rate is expected.
> match rate on movies with valid budget+revenue (after section 4) will be much higher.

In [ ]:
## 3.1 merge tmdb + crew (directors & writers)
# tmdb is the base table; crew adds director/writer nconst lists per title
base = tmdb.copy()
n_base = len(base)

crew_small = crew[["tconst", "directors", "writers"]].copy()

# left join so all tmdb rows are kept; unmatched rows get NaN for crew columns
merged = base.merge(crew_small, how="left", left_on="imdb_id", right_on="tconst")
merged.drop(columns=["tconst"], inplace=True)  # redundant after join

crew_match = merged["directors"].notna().mean()
print(f"tmdb rows:        {n_base:,}")
print(f"after crew join:  {len(merged):,}")
print(f"crew match rate:  {crew_match:.2%}")  # low overall because ~53% of tmdb rows have no imdb_id

## 3.2 merge top-billed cast from principals
# principals is many-to-one (many cast members per title), so we aggregate first
# keep only actors/actresses ranked 1-3 to avoid exploding rows on join
cast = (
    principals[principals["category"].isin(["actor", "actress"])]
    .sort_values("ordering")
    .groupby("tconst")
    .head(3)
)

# pivot long → wide so each title gets one row with cast_1, cast_2, cast_3 nconst ids
cast["rank"] = cast.groupby("tconst").cumcount() + 1
cast_wide = (
    cast.pivot(index="tconst", columns="rank", values="nconst")
    .rename(columns={1: "cast_1", 2: "cast_2", 3: "cast_3"})
    .reset_index()
)

merged = merged.merge(cast_wide, how="left", left_on="imdb_id", right_on="tconst")
merged.drop(columns=["tconst"], inplace=True)

cast_match = merged["cast_1"].notna().mean()
print(f"after cast join:  {len(merged):,}")
print(f"cast match rate:  {cast_match:.2%}")  # ~81% among rows with imdb_id

## 3.3 resolve nconst → name for director and top cast
# nconst are opaque ids; map them to human-readable names for EDA and features
name_map = names.set_index("nconst")["primaryName"]

def resolve_names(nconst_series):
    return nconst_series.map(name_map)

# directors field can contain a comma-separated list; take only the first (primary) director
merged["director_name"] = resolve_names(merged["directors"].str.split(",").str[0])
merged["cast_1_name"]   = resolve_names(merged["cast_1"])
merged["cast_2_name"]   = resolve_names(merged["cast_2"])
merged["cast_3_name"]   = resolve_names(merged["cast_3"])

print(f"\nfinal shape: {merged.shape}")
merged[["title", "imdb_id", "director_name", "cast_1_name", "cast_2_name", "cast_3_name"]].head()

In [ ]:
# diagnose: how many tmdb rows actually have an imdb_id?
has_imdb = tmdb["imdb_id"].notna() & (tmdb["imdb_id"] != "")
print(f"tmdb rows with imdb_id:    {has_imdb.sum():,} / {len(tmdb):,} ({has_imdb.mean():.2%})")

# of those, how many matched crew?
has_imdb_merged = merged["imdb_id"].notna() & (merged["imdb_id"] != "")
crew_matched = merged["directors"].notna()
print(f"crew match rate (with id): {(crew_matched & has_imdb_merged).sum() / has_imdb_merged.sum():.2%}")

cast_matched = merged["cast_1"].notna()
print(f"cast match rate (with id): {(cast_matched & has_imdb_merged).sum() / has_imdb_merged.sum():.2%}")

## 4) data cleaning

- remove invalid/missing budget or revenue rows
- parse dates and normalize schema
- define train-ready base table

In [ ]:
# cleaning scaffold
df = merged.copy()

df = df[df["budget"].fillna(0) > 0]
df = df[df["revenue"].fillna(0) > 0]

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df = df.dropna(subset=["release_date"])

# targets
df["y_log_revenue"] = np.log1p(df["revenue"])
df["profitable"] = (df["revenue"] > 1.5 * df["budget"]).astype(int)

print(df.shape)
df[["budget", "revenue", "y_log_revenue", "profitable"]].head()

## 5) feature engineering

### 5.1 group 1: structured metadata
- runtime, language, release timing, production info

### 5.2 group 2: creative content
- genre indicators (handcrafted)
- synopsis embeddings (transfer learning)

### 5.3 group 3: talent features
- director/cast history with strict temporal cutoff
- poster embeddings (transfer learning, image modality)

In [ ]:
# feature blocks scaffold
feature_blocks = {
    "g1_structured": [],
    "g2_genre_handcrafted": [],
    "g2_synopsis_embeddings": [],
    "g3_talent_temporal": [],
    "g3_poster_embeddings": []
}

# placeholder: fill each block and then concatenate into design matrices
# X_g1 = ...
# X_g2 = ...
# X_g3 = ...

## 6) data leakage analysis

- define pre-release information boundary
- explicitly exclude post-release signals from main training (e.g., ratings/votes)
- run a separate leakage demonstration experiment

## 7) ablation study

- fixed split and evaluation protocol
- run experiment matrix (e0-e8)
- compare modality contributions and interactions

## 8) fine-tuning experiment

- fine-tune one transfer model variant and compare against frozen baseline

## 9) interpretability with shap

- explain best model globally and locally

## 10) evaluation metrics

- regression: rmse, mae, r2
- optional classification view for profitability: precision/recall/f1/auc

## 11) recommended models + hyperparameter tuning

- baseline linear/ridge
- tree ensembles (rf/xgboost/lightgbm/catboost if available)
- tuned best candidate

## 12) pipeline summary and timeline

- final pipeline diagram/table
- weekly execution plan

## 13) report-ready outputs

- export key figures/tables for final report

## 14) pitfalls checklist

- no leakage
- temporal consistency for talent features
- reproducible splits and seeds